# Tests de l'API FastAPI

Ce notebook teste l'API déployée en ligne sur le Space Hugging Face.

Endpoints testés :

- `GET /api/health`
- `GET /api/metadata`
- `POST /api/predict`
- `POST /api/recommend`


## Configuration

Le notebook pointe par défaut vers l'URL publique du Space :

- `https://stephmnt-rendement-agricole.hf.space/api`

Si le Space utilise un autre domaine public, il suffit de modifier `API_BASE_URL` dans la cellule suivante.


In [1]:
import pandas as pd
import requests

SPACE_BASE_URL = "https://stephmnt-rendement-agricole.hf.space"
API_BASE_URL = f"{SPACE_BASE_URL}/api"
REQUEST_TIMEOUT_SECONDS = 30
HECTARES = 12.5

print(f"API testee : {API_BASE_URL}")


API testee : https://stephmnt-rendement-agricole.hf.space/api


## Fonctions utilitaires HTTP

Les fonctions ci-dessous interrogent directement l'API publique du Space.


In [2]:
def get_json(endpoint, params=None):
    url = f"{API_BASE_URL}{endpoint}"
    try:
        response = requests.get(url, params=params, timeout=REQUEST_TIMEOUT_SECONDS)
        response.raise_for_status()
    except requests.RequestException as exc:
        raise RuntimeError(f"Impossible de joindre l'API en ligne sur {url}: {exc}") from exc
    return response.status_code, response.json()

def post_json(endpoint, payload):
    url = f"{API_BASE_URL}{endpoint}"
    try:
        response = requests.post(url, json=payload, timeout=REQUEST_TIMEOUT_SECONDS)
        response.raise_for_status()
    except requests.RequestException as exc:
        raise RuntimeError(f"Impossible de joindre l'API en ligne sur {url}: {exc}") from exc
    return response.status_code, response.json()


## Tests de disponibilité et métadonnées


In [3]:
health_status, health_response = get_json('/health')

assert health_status == 200
assert health_response['status'] == 'ok'
assert health_response['model_source'] in {'artifact', 'fallback-trained'}

metadata_status, metadata_response = get_json('/metadata')

assert metadata_status == 200
assert metadata_response['areas']
assert metadata_response['crops']
assert 'current_year' in metadata_response
assert 'default_context' in metadata_response

preferred_area = 'France' if 'France' in metadata_response['areas'] else metadata_response['areas'][0]
preferred_crop = 'Maize' if 'Maize' in metadata_response['crops'] else metadata_response['crops'][0]

area_metadata_status, area_metadata = get_json('/metadata', params={'area': preferred_area})
assert area_metadata_status == 200

{
    'health': health_response,
    'selected_area': preferred_area,
    'selected_crop': preferred_crop,
    'current_year': area_metadata['current_year'],
    'default_context': area_metadata['default_context'],
}


{'health': {'status': 'ok', 'model_source': 'fallback-trained'},
 'selected_area': 'France',
 'selected_crop': 'Maize',
 'current_year': 2026,
 'default_context': {'average_rain_fall_mm_per_year': 867.0,
  'pesticides_tonnes': 78577.0,
  'avg_temp': 11.505}}

## Construction des payloads


In [4]:
default_context = area_metadata['default_context']
current_year = int(area_metadata['current_year'])
candidate_crops = [crop for crop in ['Maize', 'Wheat', 'Rice, paddy'] if crop in area_metadata['crops']]
if not candidate_crops:
    candidate_crops = area_metadata['crops'][:3]

predict_payload = {
    'area': preferred_area,
    'crop': preferred_crop,
    'year': current_year,
    'average_rain_fall_mm_per_year': float(default_context['average_rain_fall_mm_per_year']),
    'pesticides_tonnes': float(default_context['pesticides_tonnes']),
    'avg_temp': float(default_context['avg_temp']),
}

recommend_payload = {
    'area': preferred_area,
    'year': current_year,
    'average_rain_fall_mm_per_year': float(default_context['average_rain_fall_mm_per_year']),
    'pesticides_tonnes': float(default_context['pesticides_tonnes']),
    'avg_temp': float(default_context['avg_temp']),
    'candidate_crops': candidate_crops,
}

{
    'predict_payload': predict_payload,
    'recommend_payload': recommend_payload,
}


{'predict_payload': {'area': 'France',
  'crop': 'Maize',
  'year': 2026,
  'average_rain_fall_mm_per_year': 867.0,
  'pesticides_tonnes': 78577.0,
  'avg_temp': 11.505},
 'recommend_payload': {'area': 'France',
  'year': 2026,
  'average_rain_fall_mm_per_year': 867.0,
  'pesticides_tonnes': 78577.0,
  'avg_temp': 11.505,
  'candidate_crops': ['Maize', 'Wheat', 'Rice, paddy']}}

## Test de `POST /predict`


In [5]:
predict_status, predict_response = post_json('/predict', predict_payload)

assert predict_status == 200
assert 'predicted_yield_t_ha' in predict_response
assert float(predict_response['predicted_yield_t_ha']) >= 0.0

{
    'payload': predict_payload,
    'response': predict_response,
}


{'payload': {'area': 'France',
  'crop': 'Maize',
  'year': 2026,
  'average_rain_fall_mm_per_year': 867.0,
  'pesticides_tonnes': 78577.0,
  'avg_temp': 11.505},
 'response': {'predicted_yield_t_ha': 12.0559}}

## Test de `POST /recommend`


In [6]:
recommend_status, recommend_response = post_json('/recommend', recommend_payload)
recommendations = recommend_response['recommendations']

assert recommend_status == 200
assert len(recommendations) > 0
assert recommendations == sorted(
    recommendations,
    key=lambda row: row['predicted_yield_t_ha'],
    reverse=True,
)

recommendations_df = pd.DataFrame(recommendations)
recommendations_df['predicted_total_production_tons'] = recommendations_df['predicted_yield_t_ha'] * HECTARES
recommendations_df


,crop,predicted_yield_t_ha,predicted_total_production_tons
0,"Rice, paddy",12.3596,154.49500
1,Maize,12.0559,150.69875
2,Wheat,9.7466,121.83250
